# Recursion Foundations and Call Stack Mechanics

**Session:** 2026-11-02 · Recursion and Two Pointer  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

Recursion is a fundamental problem-solving paradigm in computer science where a function solves a problem by calling itself with smaller subproblems. Every well-formed recursive procedure consists of two essential components:
1. **Base Case(s):** One or more terminal conditions that can be evaluated directly without further recursive calls, preventing infinite descent and stack exhaustion.
2. **Recursive Case(s):** Decomposing the current problem instance into one or more strictly smaller instances of the same problem, invoking the function on the smaller instances, and combining their results.


## 1. Linear Iteration vs Linear Recursion: Sum of a List

Consider computing the sum of elements in an array:
$$\text{total} = \sum_{i=0}^{n-1} a_i$$

In an iterative approach, an accumulator variable is updated sequentially across a single loop pass. In a recursive approach, the sum of a list equals its first element plus the sum of the remaining suffix:
$$\text{sum}(\langle a_0, a_1, \dots, a_{n-1} \rangle) = a_0 + \text{sum}(\langle a_1, \dots, a_{n-1} \rangle)$$
with the base case $\text{sum}(\langle \rangle) = 0$.


In [ ]:
from collections.abc import Sequence


def sum_list_iterative(arr: Sequence[int | float]) -> int | float:
    """Compute sum of an array iteratively.

    Time Complexity: O(n)
    Auxiliary Space Complexity: O(1)
    """
    total = 0
    for val in arr:
        total += val
    return total


# Verification on lecture sample
arr_sample = [1, 2, 1, 5, 3]
print("Iterative sum:", sum_list_iterative(arr_sample))
assert sum_list_iterative(arr_sample) == 12

## 2. Recursive Sum: Slicing vs Index Passing

A common pitfall in Python is using list slicing `arr[1:]` in recursion. Slicing allocates a new copy of the remaining $n-1$ elements. Therefore, summing an array of length $n$ via slicing performs:
$$\sum_{k=1}^{n} O(k) = O(n^2)$$
total copying operations!

To achieve true $O(n)$ time complexity, we pass a pointer/index $i$ representing the current position without slicing.


In [ ]:
def sum_list_recursive_slicing(arr: list[int | float]) -> int | float:
    """Compute sum using list slicing.

    Time Complexity: O(n^2) due to slice duplication
    Auxiliary Space: O(n) call stack + O(n^2) heap allocations
    """
    if not arr:
        return 0
    return arr[0] + sum_list_recursive_slicing(arr[1:])


def sum_list_recursive_indexed(arr: Sequence[int | float], index: int = 0) -> int | float:
    """Compute sum using an index pointer to eliminate memory copying.

    Time Complexity: O(n)
    Auxiliary Space: O(n) call stack frames
    """
    if index == len(arr):
        return 0
    return arr[index] + sum_list_recursive_indexed(arr, index + 1)


print("Recursive slicing sum:", sum_list_recursive_slicing(arr_sample))
print("Recursive indexed sum:", sum_list_recursive_indexed(arr_sample))
assert sum_list_recursive_slicing(arr_sample) == 12
assert sum_list_recursive_indexed(arr_sample) == 12

## 3. The Factorial Function and Call Stack Mechanics

For any integer $n \ge 0$, the factorial function $n!$ is defined by:
$$n! = \begin{cases} 1 & \text{if } n = 0 \\ n \times (n-1)! & \text{if } n \ge 1 \end{cases}$$

When `factorial(4)` is called:
1. `factorial(4)` pushes frame to stack, requires `factorial(3)`
2. `factorial(3)` pushes frame, requires `factorial(2)`
3. `factorial(2)` pushes frame, requires `factorial(1)`
4. `factorial(1)` pushes frame, requires `factorial(0)`
5. `factorial(0)` hits base case, returns `1`
6. Unwinding:
   - `1 * 1 = 1`
   - `2 * 1 = 2`
   - `3 * 2 = 6`
   - `4 * 6 = 24`


In [ ]:
from collections.abc import Callable
from functools import wraps
from typing import Any


def trace_recursion(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator that traces recursive call activations and returns."""
    depth = 0

    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        nonlocal depth
        indent = "  " * depth
        arg_repr = ", ".join([repr(a) for a in args])
        print(f"{indent}-> {func.__name__}({arg_repr})")
        depth += 1
        result = func(*args, **kwargs)
        depth -= 1
        print(f"{indent}<- {func.__name__}({arg_repr}) = {result}")
        return result

    return wrapper


def factorial(n: int) -> int:
    """Compute n! with input validation and O(n) time/space."""
    if n < 0:
        raise ValueError("Factorial is undefined for negative integers.")
    if n == 0:
        return 1
    return n * factorial(n - 1)


@trace_recursion
def factorial_traced(n: int) -> int:
    if n == 0:
        return 1
    return n * factorial_traced(n - 1)


print("--- Trace of factorial(4) ---")
result = factorial_traced(4)
print("Final result:", result)
assert result == 24
assert factorial(0) == 1
assert factorial(1) == 1
assert factorial(5) == 120

## 4. Call Stack Limits and Memory Guardrails

Python sets a recursion depth limit (accessible via `sys.getrecursionlimit()`, typically 1000) to protect the C-level stack from buffer overflows. Exceeding this limit raises `RecursionError`.


In [ ]:
import sys

print("Current recursion limit:", sys.getrecursionlimit())

# Safe boundary verification
try:

    def overflow_test(k: int) -> int:
        if k > sys.getrecursionlimit() + 50:
            return k
        return overflow_test(k + 1)

    overflow_test(1)
except RecursionError as e:
    print("Caught expected RecursionError:", type(e).__name__)